# Coretan, Tanya-Jawab, & Eksplorasi Komputasi Persiapan UTS
## Mata Kuliah MA4151 Kriptografi — Institut Teknologi Bandung
---
Buku catatan interaktif ini (**`coret.ipynb`**) dibuat khusus untuk:
1. **Mendokumentasikan pertanyaan dan pembahasan tanya-jawab** persiapan UTS secara terstruktur dan matematis.
2. **Menyediakan mesin komputasi kriptografi** (modulo arithmetic, kriptoanalisis, Teori Shannon, DES/AES, teori bilangan, RSA, dan titik tetap).
3. **Memverifikasi perhitungan manual** pada soal-soal latihan dan ujian tahun-tahun sebelumnya.


## 📌 Bagian 1: Mesin Aritmetika Modular & Teori Bilangan Inti

Komponen dasar yang digunakan di seluruh algoritma kriptografi klasik dan modern:
- FPB & Algoritma Euclidean Diperluas (**Extended Euclidean Algorithm - EEA**)
- Invers modular $a^{-1} \pmod m$
- Penyelesaian kongruensi linier $ax \equiv b \pmod m$
- Teorema Sisa Cina (**Chinese Remainder Theorem - CRT**) umum
- Fungsi Phi Euler $\varphi(n)$ dan faktorisasi prima


In [ ]:
import math
from functools import reduce

def egcd(a, b):
    """Algoritma Euclidean Diperluas: mengembalikan (gcd, x, y) sedemikian hingga ax + by = gcd."""
    if a == 0:
        return b, 0, 1
    g, y, x = egcd(b % a, a)
    return g, x - (b // a) * y, y

def modinv(a, m):
    """Menghitung invers multiplikatif a modulo m: a^(-1) mod m."""
    g, x, _ = egcd(a % m, m)
    if g != 1:
        raise ValueError(f"Invers tidak ada karena gcd({a}, {m}) = {g} != 1")
    return x % m

def solve_linear_congruence(a, b, m):
    """Menyelesaikan ax = b (mod m). Mengembalikan daftar seluruh solusi unik modulo m."""
    a = a % m
    b = b % m
    g = math.gcd(a, m)
    if b % g != 0:
        return [] # Tidak ada solusi
    # Sederhanakan persamaan
    a_red, b_red, m_red = a // g, b // g, m // g
    x0 = (b_red * modinv(a_red, m_red)) % m_red
    return [(x0 + k * m_red) % m for k in range(g)]

def crt(remainders, moduli):
    """Chinese Remainder Theorem untuk modulus yang sepasang-sepasang relatif prima."""
    total = 0
    M = reduce(lambda a, b: a * b, moduli)
    for a_i, m_i in zip(remainders, moduli):
        M_i = M // m_i
        y_i = modinv(M_i, m_i)
        total += a_i * M_i * y_i
    return total % M, M

def prime_factors(n):
    """Mengembalikan kamus faktor prima dan eksponennya: {p: k}."""
    factors = {}
    d = 2
    temp = n
    while d * d <= temp:
        if temp % d == 0:
            count = 0
            while temp % d == 0:
                count += 1
                temp //= d
            factors[d] = count
        d = 3 if d == 2 else d + 2
    if temp > 1:
        factors[temp] = 1
    return factors

def phi(n):
    """Fungsi Phi Euler phi(n)."""
    factors = prime_factors(n)
    res = n
    for p in factors:
        res = res * (p - 1) // p
    return res

print("=== Tes Mesin Teori Bilangan ===")
print("Invers dari 7 mod 26:", modinv(7, 26))
print("phi(26):", phi(26), "| phi(24):", phi(24), "| phi(1800):", phi(1800))
sol_crt, mod_crt = crt([2, 3, 2], [3, 5, 7])
print(f"Solusi x = 2 mod 3, 3 mod 5, 2 mod 7 -> x = {sol_crt} (mod {mod_crt})")


## 📌 Bagian 2: Sandi Klasik, Ruang ($P, C, K$), & Keamanan Sempurna

Di bagian ini kita dapat:
1. Menghitung ukuran ruang $\mathcal{P}, \mathcal{C}, \mathcal{K}$ untuk berbagai sandi klasik.
2. Memverifikasi secara eksperimental mengapa **Sandi Hill tidak aman sempurna** (karena vektor nol selalu dipetakan ke vektor nol).
3. Menganalisis tabrakan teks-sandi (**ciphertext collision**) pada matriks Hill yang determinannya non-invertible modulo 26.


In [ ]:
# 1. Ruang Kunci Eksak Sandi Hill m x m
def hill_keyspace_size(m):
    """Ukuran ruang kunci GL(m, Z_26) = |GL(m, F_2)| * |GL(m, F_13)|"""
    f2_size = 1
    for i in range(m):
        f2_size *= (2**m - 2**i)
    f13_size = 1
    for i in range(m):
        f13_size *= (13**m - 13**i)
    return f2_size * f13_size

print("Ukuran Keyspace Sandi Hill 2x2:", hill_keyspace_size(2))
print("Ukuran Keyspace Sandi Hill 3x3:", hill_keyspace_size(3))

# 2. Verifikasi Tabrakan Sandi Hill Non-Invertible (Kasus Soal UTS 2021-2022)
# Alice memilih matriks K = [[1, 2], [3, 4]].
# det(K) = 1*4 - 2*3 = -2 = 24 = 0 mod 2 -> TIDAK INVERTIBLE!
import numpy as np

K = np.array([[1, 2], [3, 4]])
det_K = int(round(np.linalg.det(K))) % 26
print(f"det(K) mod 26 = {det_K}, gcd(det(K), 26) = {math.gcd(det_K, 26)}")

# Mencari vektor nol-space tak nol v sedemikian hingga K * v = [0, 0] mod 26
null_vectors = []
for x in range(26):
    for y in range(26):
        if x == 0 and y == 0:
            continue
        v = np.array([x, y])
        res = np.dot(K, v) % 26
        if res[0] == 0 and res[1] == 0:
            null_vectors.append((x, y))

print(f"Ditemukan {len(null_vectors)} vektor selisih plaintext yang menghasilkan ciphertext identik!")
print("Contoh vektor tabrakan v:", null_vectors[:5])
v_sample = np.array(null_vectors[0])
p1 = np.array([5, 8]) # contoh plaintext 'FI'
p2 = (p1 + v_sample) % 26
c1 = np.dot(K, p1) % 26
c2 = np.dot(K, p2) % 26
print(f"Plaintext 1: {p1} -> Ciphertext 1: {c1}")
print(f"Plaintext 2: {p2} -> Ciphertext 2: {c2}")
print("Apakah ciphertext sama persis?", np.array_equal(c1, c2))


## 📌 Bagian 3: Kriptoanalisis (KPA Affine, Uji Kasiski, & Indeks Koinsidensi)

Alat analisis frekuensi dan pemecahan teks sandi:
- **Solver KPA Sandi Affine**: memulihkan $(a, b)$ dari 2 pasang korespondensi karakter.
- **Detektor Uji Kasiski**: mencari substring berulang panjang $\ge 3$ dan menghitung $\gcd$ jaraknya.
- **Kalkulator Indeks Koinsidensi ($I_c$)**: rumus $\frac{\sum f_i(f_i-1)}{N(N-1)}$.
- **Estimasi Periode Kunci Friedman**: $m \approx \frac{\kappa_p - \kappa_r}{I_c - \kappa_r + \frac{1}{N}(\kappa_p - I_c)}$.


In [ ]:
# 1. Solver KPA Affine
def solve_kpa_affine(p1, c1, p2, c2):
    """Mencari (a, b) dari p1 -> c1 dan p2 -> c2 modulo 26."""
    delta_p = (p1 - p2) % 26
    delta_c = (c1 - c2) % 26
    g = math.gcd(delta_p, 26)
    if g != 1:
        return f"Gagal: gcd(delta_p, 26) = {g} != 1"
    a = (delta_c * modinv(delta_p, 26)) % 26
    b = (c1 - a * p1) % 26
    return {'a': a, 'b': b}

# Contoh: 'hahaha' -> 'NONONO' (Soal UTS 2020-2021 No 4)
# 'h' = 7, 'a' = 0, 'N' = 13, 'O' = 14
# p1 = 7 ('h') -> c1 = 13 ('N')
# p2 = 0 ('a') -> c2 = 14 ('O')
kunci_affine = solve_kpa_affine(7, 13, 0, 14)
print("Kunci Affine (m, n) dari 'hahaha' -> 'NONONO':", kunci_affine)
print("Nilai m + n:", kunci_affine['a'] + kunci_affine['b'])

# 2. Uji Kasiski
from collections import defaultdict

def kasiski_test(ciphertext, min_len=3):
    text = ''.join([c.upper() for c in ciphertext if c.isalpha()])
    patterns = defaultdict(list)
    for length in range(min_len, min_len + 3):
        for i in range(len(text) - length + 1):
            sub = text[i:i+length]
            patterns[sub].append(i)
    
    repeated = {k: v for k, v in patterns.items() if len(v) > 1}
    diffs = []
    for sub, positions in repeated.items():
        sub_diffs = [positions[i+1] - positions[i] for i in range(len(positions)-1)]
        diffs.extend(sub_diffs)
    
    overall_gcd = reduce(math.gcd, diffs) if diffs else 1
    return repeated, diffs, overall_gcd

# Uji Kasiski Soal UTS 2024-2025 No 5:
# Ciphertext: KXQMYTOZKQWHWMEKXQQXKQXYM
ct_uts24 = "KXQMYTOZKQWHWMEKXQQXKQXYM"
rep, diffs, g = kasiski_test(ct_uts24, min_len=3)
print("Pola berulang Kasiski UTS 2024:")
for k, v in rep.items():
    print(f"  Substring '{k}' di indeks {v}, selisih: {[v[i+1]-v[i] for i in range(len(v)-1)]}")
print("Dugaan pembagi panjang kunci (GCD selisih):", g)

# 3. Indeks Koinsidensi (IC)
def index_of_coincidence(text):
    text = ''.join([c.upper() for c in text if c.isalpha()])
    N = len(text)
    if N <= 1:
        return 0.0
    counts = [text.count(chr(ord('A') + i)) for i in range(26)]
    sum_f = sum(f * (f - 1) for f in counts)
    return sum_f / (N * (N - 1))

print("IC teks UTS 2024:", index_of_coincidence(ct_uts24))


## 📌 Bagian 4: Teori Shannon, Entropi, & Pengkodean Huffman

Topik wajib ujian:
- Perhitungan **Entropi Shannon $H(X) = -\sum p(x) \log_2 p(x)$**.
- Pembentukan **Pohon Huffman (Prefix-Free Code)** optimal secara otomatis.
- Evaluasi panjang rata-rata ekspektasi $L(X) = \sum p(x_i) \ell(x_i)$, efisiensi $\eta = \frac{H(X)}{L(X)}$, dan redundansi $\bar{R} = L(X) - H(X)$.


In [ ]:
import heapq

class HuffmanNode:
    def __init__(self, char, prob):
        self.char = char
        self.prob = prob
        self.left = None
        self.right = None
    
    def __lt__(self, other):
        return self.prob < other.prob

def build_huffman_tree(dist):
    """dist: dict {simbol: probabilitas}"""
    heap = [HuffmanNode(char, prob) for char, prob in dist.items()]
    heapq.heapify(heap)
    
    while len(heap) > 1:
        node1 = heapq.heappop(heap)
        node2 = heapq.heappop(heap)
        
        merged = HuffmanNode(None, node1.prob + node2.prob)
        merged.left = node1
        merged.right = node2
        heapq.heappush(heap, merged)
        
    return heap[0]

def get_codewords(root, prefix="", code_map=None):
    if code_map is None:
        code_map = {}
    if root is not None:
        if root.char is not None:
            code_map[root.char] = prefix
        get_codewords(root.left, prefix + "0", code_map)
        get_codewords(root.right, prefix + "1", code_map)
    return code_map

def evaluate_huffman(dist):
    root = build_huffman_tree(dist)
    codes = get_codewords(root)
    
    # Hitung Entropi Shannon H(X)
    H_X = -sum(p * math.log2(p) for p in dist.values() if p > 0)
    
    # Hitung L(X)
    L_X = sum(dist[char] * len(codes[char]) for char in dist)
    
    efficiency = (H_X / L_X) * 100
    redundancy = L_X - H_X
    
    return {
        'codes': codes,
        'H(X)': H_X,
        'L(X)': L_X,
        'redundancy': redundancy,
        'efficiency (%)': efficiency
    }

# Evaluasi Distribusi Soal UTS 2022-2023 & 2023-2024:
# p(a) = 0.35, p(b) = 0.25, p(c) = 0.20, p(d) = 0.12, p(e) = 0.08
dist_uts22 = {'a': 0.35, 'b': 0.25, 'c': 0.20, 'd': 0.12, 'e': 0.08}
res22 = evaluate_huffman(dist_uts22)

print("=== Hasil Pengkodean Huffman Soal UTS 2022/2023 ===")
print("Codewords:", res22['codes'])
print(f"Entropi H(X)        : {res22['H(X)']:.4f} bit/simbol")
print(f"Panjang Rata-rata L : {res22['L(X)']:.4f} bit/simbol")
print(f"Redundansi L - H(X) : {res22['redundancy']:.4f} bit")
print(f"Efisiensi           : {res22['efficiency (%)']:.2f}%")
print("Memenuhi Teorema Shannon H(X) <= L(X) < H(X)+1:", res22['H(X)'] <= res22['L(X)'] < res22['H(X)'] + 1)

# Evaluasi Distribusi Soal UTS 2024-2025:
# p(a)=0.2, p(b)=0.35, p(c)=0.17, p(d)=0.03, p(e)=0.25
dist_uts24 = {'a': 0.20, 'b': 0.35, 'c': 0.17, 'd': 0.03, 'e': 0.25}
res24 = evaluate_huffman(dist_uts24)
print("\n=== Hasil Pengkodean Huffman Soal UTS 2024/2025 ===")
print("Codewords:", res24['codes'])
print(f"Entropi H(X)        : {res24['H(X)']:.4f} bit/simbol")
print(f"Panjang Rata-rata L : {res24['L(X)']:.4f} bit/simbol")


## 📌 Bagian 5: Block Cipher Modern (DES & AES Scratchpad)

Komputasi aljabar lapangan biner $GF(2^8)$ pada AES:
- Polinomial tak tereduksi: $m(x) = x^8 + x^4 + x^3 + x + 1$ (`0x11B`)
- Operasi `xtime(b)` (perkalian dengan $\{02\}$)
- Perkalian dua byte sembarang di $GF(2^8)$
- Verifikasi matriks transformasi **MixColumns** dengan sukubanyak $\{03\}z^3 + \{01\}z^2 + \{01\}z + \{02\}$


In [ ]:
def xtime(b):
    """Perkalian dengan 0x02 dalam GF(2^8) modulo 0x11B."""
    res = (b << 1) & 0xFF
    if b & 0x80:
        res ^= 0x1B
    return res

def gf_mul(a, b):
    """Perkalian dua byte sembarang di GF(2^8)."""
    res = 0
    temp_a = a
    for i in range(8):
        if (b >> i) & 1:
            res ^= temp_a
        temp_a = xtime(temp_a)
    return res

print("=== Tes Aljabar AES GF(2^8) ===")
print("xtime(0x57) =", hex(xtime(0x57)))
print("gf_mul(0x57, 0x02) =", hex(gf_mul(0x57, 0x02)))
print("gf_mul(0x57, 0x13) =", hex(gf_mul(0x57, 0x13)))

# MixColumns satu kolom state (vektor 4 byte)
def mix_column(col):
    """Perkalian kolom dengan matriks sirkulan MixColumns."""
    s0, s1, s2, s3 = col
    r0 = gf_mul(0x02, s0) ^ gf_mul(0x03, s1) ^ s2 ^ s3
    r1 = s0 ^ gf_mul(0x02, s1) ^ gf_mul(0x03, s2) ^ s3
    r2 = s0 ^ s1 ^ gf_mul(0x02, s2) ^ gf_mul(0x03, s3)
    r3 = gf_mul(0x03, s0) ^ s1 ^ s2 ^ gf_mul(0x02, s3)
    return [r0, r1, r2, r3]

sample_col = [0xDB, 0x13, 0x53, 0x45]
res_col = mix_column(sample_col)
print(f"MixColumns {[hex(x) for x in sample_col]} -> {[hex(x) for x in res_col]}")


## 📌 Bagian 6: RSA & Analisis Titik Tetap (Fixed Points)

Pembuktian Teorema Titik Tetap RSA:
Pesan $x \in \mathbb{Z}_n^*$ memenuhi $e_K(x) \equiv x^b \equiv x \pmod n$ jika dan hanya jika:
$$N_{\text{tetap}} = \gcd(b - 1, p - 1) \cdot \gcd(b - 1, q - 1)$$

Di bawah ini kode untuk menghitung rumus eksak dan memverifikasinya secara langsung melalui pencarian tuntas (*brute-force search*) pada seluruh $x \in \mathbb{Z}_n^*$.


In [ ]:
def rsa_fixed_points_formula(p, q, b):
    """Menghitung jumlah titik tetap berdasarkan rumus teorema."""
    return math.gcd(b - 1, p - 1) * math.gcd(b - 1, q - 1)

def rsa_fixed_points_bruteforce(p, q, b):
    """Mencari seluruh teks asal tetap x in Z_n^* sedemikian hingga x^b = x mod n."""
    n = p * q
    fixed_points = []
    for x in range(1, n):
        if math.gcd(x, n) == 1:
            if pow(x, b, n) == x:
                fixed_points.append(x)
    return fixed_points

# Uji Coba Numerik
p_test, q_test, b_test = 11, 13, 7
n_test = p_test * q_test
teori = rsa_fixed_points_formula(p_test, q_test, b_test)
aktual = rsa_fixed_points_bruteforce(p_test, q_test, b_test)

print(f"Parameter RSA: p={p_test}, q={q_test}, n={n_test}, b={b_test}")
print("Jumlah titik tetap menurut rumus Teorema:", teori)
print("Jumlah titik tetap menurut pencarian tuntas:", len(aktual))
print("Apakah rumus terbukti akurat 100%?", teori == len(aktual))
print("Daftar pesan tetap:", aktual)


### ❓ Pertanyaan 1: Apa yang Dimaksud dengan "Blok" dalam Kriptografi?
---

### 1. Definisi Formal & Konseptual
Dalam kriptografi, **blok (*block*)** adalah **sekumpulan elemen berurutan berpanjang tetap (*fixed-length sequence*)**—baik berupa karakter alfabet, byte, maupun bit biner—yang **diperlakukan, dienkripsi, dan didekripsi secara bersama-sama sebagai satu kesatuan tunggal (*single atomic unit*)**.

Panjang dari kumpulan elemen tersebut disebut **ukuran blok (*block size*)**, yang umumnya dilambangkan dengan:
- $m$ karakter alfabet pada sistem kripto klasik (misalnya $m = 2$ atau $m = 3$ pada Sandi Hill dan Sandi Permutasi).
- $n$ bit (atau $B = n/8$ byte) pada sistem kripto modern (misalnya $n = 64$ bit pada DES, dan $n = 128$ bit pada AES).

Jika panjang teks asli tidak habis dibagi oleh ukuran blok, pesan harus dilengkapi menggunakan teknik penambahan bit/karakter pengganjal (**padding**).

---

### 2. Perbedaan Paradigma: Cipher Blok (*Block Cipher*) vs. Cipher Alir (*Stream Cipher*)

| Parameter Pembanding | Cipher Blok (*Block Cipher*) | Cipher Alir (*Stream Cipher*) |
| :--- | :--- | :--- |
| **Unit Pemrosesan** | Blok data berukuran tetap $n$-bit (misal 64 bit atau 128 bit). | Simbol individual secara kontinu (1 bit atau 1 karakter) per waktu $t$. |
| **Fungsi Transformasi** | $C_i = E_K(P_i)$, pemetaan bijektif ruang blok $\mathcal{P} \to \mathcal{C}$ terindeks kunci $K$. | $c_t = p_t \oplus z_t$, kombinasi linear (XOR) dengan aliran kunci (*keystream*) $z_t$. |
| **Ketergantungan Bit** | **Tinggi (Efek Longsoran / Difusi):** 1 bit pada blok plaintext mempengaruhi seluruh bit pada blok ciphertext. | **Rendah (Lokal):** 1 bit pada plaintext hanya mempengaruhi 1 bit pada ciphertext yang bersesuaian. |
| **Kebutuhan Padding** | **Wajib:** Jika panjang teks bukan kelipatan ukuran blok, butuh padding (misal PKCS#7). | **Tidak Butuh:** Panjang ciphertext selalu tepat sama dengan panjang plaintext. |
| **Contoh Sistem** | DES (64-bit), AES (128-bit), Sandi Hill ($m$-karakter). | LFSR, RC4, ChaCha20, One-Time Pad. |

---

### 3. Peranan Blok dalam Kriptosistem Klasik

1. **Sandi Hill ($m \times m$):**
   - Teks dibagi menjadi blok berukuran $m$ huruf, dipandang sebagai vektor kolom $\mathbf{x} = (x_1, x_2, \dots, x_m)^T \in \mathbb{Z}_{26}^m$.
   - Enkripsi: $\mathbf{y} \equiv K \mathbf{x} \pmod{26}$.
   - **Tujuan Blok:** Menghancurkan analisis frekuensi unigram! Huruf yang sama (misalnya huruf `'A'`) akan dienkripsi menjadi huruf sandi yang berbeda tergantung pada kombinasi huruf-huruf lain di dalam blok yang sama.

2. **Sandi Permutasi (Transposisi Blok $m$):**
   - Teks dibagi menjadi blok $m$ huruf: $\mathbf{x} = (x_1, \dots, x_m)$.
   - Enkripsi mengacak posisi karakter di dalam batas masing-masing blok sesuai permutasi $\pi \in S_m$: $e_\pi(\mathbf{x}) = (x_{\pi(1)}, \dots, x_{\pi(m)})$.

3. **Sandi Vigenère:**
   - Secara inheren memproses teks dalam blok-blok periodik berukuran $m$ (panjang kata kunci), di mana huruf ke-$j$ pada blok selalu digeser dengan subkunci $k_j$.

---

### 4. Peranan Blok dalam Kriptosistem Modern

1. **Data Encryption Standard (DES):**
   - Ukuran blok: **$n = 64$ bit** (8 byte).
   - Di dalam Jaringan Feistel 16 ronde, blok 64-bit ini dibagi simetris menjadi dua blok paruh: $L_i$ (32 bit) dan $R_i$ (32 bit).

2. **Advanced Encryption Standard (AES / Rijndael):**
   - Ukuran blok: **$n = 128$ bit** (16 byte), baik untuk AES-128, AES-192, maupun AES-256.
   - Blok 16 byte dipetakan ke dalam **matriks state $4 \times 4$ byte**:
     $$S = \begin{bmatrix} s_{0,0} & s_{0,1} & s_{0,2} & s_{0,3} \\ s_{1,0} & s_{1,1} & s_{1,2} & s_{1,3} \\ s_{2,0} & s_{2,1} & s_{2,2} & s_{2,3} \\ s_{3,0} & s_{3,1} & s_{3,2} & s_{3,3} \end{bmatrix}$$
     Setiap sel $s_{i,j}$ adalah elemen lapangan Galois $GF(2^8)$. Seluruh transformasi ronde (SubBytes, ShiftRows, MixColumns, AddRoundKey) beroperasi pada blok matriks state ini.

---

### 5. Mengapa Ukuran Blok Sangat Krusial? (Batas Keamanan *Birthday Bound*)

Ukuran blok menentukan ketahanan cipher terhadap serangan statistik (*collision attack*):
- Pada mode berantai seperti CBC, jika dua blok ciphertext menghasilkan tabrakan ($C_i = C_j$), maka $P_i \oplus C_{i-1} = P_j \oplus C_{j-1} \implies P_i \oplus P_j = C_{i-1} \oplus C_{j-1}$, yang membocorkan XOR dari dua blok plaintext!
- Berdasarkan **Paradoks Ulang Tahun (*Birthday Paradox*)**, tabrakan blok diperkirakan terjadi setelah mengenkripsi sekitar:
  $$2^{n/2} \text{ blok}$$
- **Pada DES ($n = 64$ bit):** Tabrakan terjadi setelah $2^{32}$ blok $\approx 4$ miliar blok $\approx 32$ GB data (inilah dasar serangan *Sweet32* yang menyebabkan DES/3DES dihentikan penggunaannya).
- **Pada AES ($n = 128$ bit):** Tabrakan baru terjadi setelah $2^{64}$ blok $\approx 1.8 \times 10^{19}$ blok $\approx 250$ Exabyte data. Sangat aman!


In [ ]:
# =========================================================================
# Simulasi & Demonstrasi Konsep Blok, Padding, dan Matriks State AES
# =========================================================================

# 1. Pemotongan Teks Menjadi Blok-Blok (Chunking)
def text_to_blocks(plaintext, block_size):
    """Membagi teks menjadi blok-blok berukuran tetap."""
    blocks = [plaintext[i:i+block_size] for i in range(0, len(plaintext), block_size)]
    return blocks

sample_text = "KRIPTOGRAFIITB"
m = 4 # ukuran blok 4 karakter
print(f"Teks Asli: '{sample_text}' (Panjang: {len(sample_text)})")
print(f"Pembagian ke Blok berukuran {m} karakter:", text_to_blocks(sample_text, m))
print(f"Catatan: Blok terakhir '{text_to_blocks(sample_text, m)[-1]}' hanya 2 huruf -> butuh padding!\n")

# 2. Skema Padding Standar Industri: PKCS#7
def pkcs7_pad(data_bytes, block_size=16):
    """Menambahkan padding PKCS#7 agar panjang data kelipatan block_size."""
    pad_len = block_size - (len(data_bytes) % block_size)
    if pad_len == 0:
        pad_len = block_size
    padding = bytes([pad_len] * pad_len)
    return data_bytes + padding

def pkcs7_unpad(padded_bytes):
    """Memvalidasi dan membuang padding PKCS#7."""
    pad_len = padded_bytes[-1]
    if pad_len == 0 or pad_len > len(padded_bytes):
        raise ValueError("Padding tidak valid!")
    if padded_bytes[-pad_len:] != bytes([pad_len] * pad_len):
        raise ValueError("Padding PKCS#7 korup!")
    return padded_bytes[:-pad_len]

raw_data = b"Ujian UTS Kripto"
print(f"Data Mentah (panjang {len(raw_data)} byte): {raw_data}")
padded = pkcs7_pad(raw_data, block_size=16)
print(f"Data Ter-pad (panjang {len(padded)} byte): {[hex(b) for b in padded]}")
print(f"Nilai padding yang ditambahkan: {list(padded[len(raw_data):])} (karena pas 16 byte, ditambahkan 1 blok penuh 16 byte bernilai 0x10)")
print(f"Data setelah di-unpad: {pkcs7_unpad(padded).decode('utf-8')}\n")

# 3. Representasi Blok AES 128-bit (16 Byte) Menjadi Matriks State 4x4
def bytes_to_aes_state(block_bytes):
    """Mengonversi 16 byte menjadi matriks state 4x4 byte (kolom per kolom)."""
    assert len(block_bytes) == 16, "Blok AES wajib tepat 16 byte (128 bit)!"
    state = [[0]*4 for _ in range(4)]
    for r in range(4):
        for c in range(4):
            state[r][c] = block_bytes[r + 4*c]
    return state

aes_block_example = bytes([0x32, 0x88, 0x31, 0xE0, 0x43, 0x5A, 0x31, 0x37,
                          0xF6, 0x30, 0x98, 0x07, 0xA8, 0x8D, 0xA2, 0x34])
state_matrix = bytes_to_aes_state(aes_block_example)
print("=== Matriks State AES (4x4 Byte) dari Blok 128-bit ===")
for row in state_matrix:
    print("[" + " ".join(f"{val:02X}" for val in row) + "]")


### ❓ Pertanyaan 2: Sandi Geser 1 Karakter, Apakah Aman Sempurna Karena $P(k) = 1/|\mathcal{K}| = 1/26$ dan Menurut Teorema Karakterisasi Shannon Termasuk Aman Sempurna?
---

### Jawaban: **YA, TEPAT SEKALI!**

Mari kita uji secara presisi berdasarkan **Teorema Karakterisasi Shannon (1949)** untuk kasus khusus ketika ukuran ruang kunci sama dengan ukuran ruang pesan dan ruang sandi:

#### 1. Teorema Karakterisasi Shannon ($|\mathcal{K}| = |\mathcal{C}| = |\mathcal{P}|$)
> Misalkan $(\mathcal{P}, \mathcal{C}, \mathcal{K}, \mathcal{E}, \mathcal{D})$ adalah suatu kriptosistem di mana $|\mathcal{K}| = |\mathcal{C}| = |\mathcal{P}|$. Kriptosistem ini memiliki **keamanan sempurna (*perfect secrecy*)** jika dan hanya jika memenuhi dua syarat berikut:
> 1. Kunci berdistribusi seragam: untuk setiap $k \in \mathcal{K}$, berlaku $P(K = k) = \frac{1}{|\mathcal{K}|}$.
> 2. Untuk setiap pasangan plaintext $x \in \mathcal{P}$ dan ciphertext $y \in \mathcal{C}$, terdapat **tepat satu (*unique*)** kunci $k \in \mathcal{K}$ sedemikian hingga $e_k(x) = y$.

#### 2. Pembuktian pada Sandi Geser 1 Karakter
- **Ukuran Himpunan:**
  $$\mathcal{P} = \mathbb{Z}_{26} \implies |\mathcal{P}| = 26$$
  $$\mathcal{C} = \mathbb{Z}_{26} \implies |\mathcal{C}| = 26$$
  $$\mathcal{K} = \mathbb{Z}_{26} \implies |\mathcal{K}| = 26$$
  Kondisi awal $|\mathcal{K}| = |\mathcal{C}| = |\mathcal{P}| = 26$ **terpenuhi**.
- **Syarat 1 (Distribusi Kunci Seragam):**
  Jika kunci $k$ dipilih secara acak murni seragam dari 26 kemungkinan, maka:
  $$P(K = k) = \frac{1}{26} = \frac{1}{|\mathcal{K}|} \quad (\text{Terpenuhi!})$$
- **Syarat 2 (Keunikan Kunci per Pasangan $(x, y)$):**
  Aturan enkripsi: $y \equiv x + k \pmod{26}$.
  Untuk sembarang huruf asal $x \in \mathbb{Z}_{26}$ dan sembarang huruf sandi $y \in \mathbb{Z}_{26}$, kunci yang menghubungkan keduanya adalah:
  $$k \equiv (y - x) \pmod{26}$$
  Karena $(\mathbb{Z}_{26}, +)$ adalah grup siklis, elemen $(y - x) \bmod 26$ selalu ada dan **bersifat tunggal (unik)** di dalam $\mathbb{Z}_{26}$! $(\text{Terpenuhi!})$

#### 3. Konsekuensi Matematis:
Karena kedua syarat terpenuhi, peluang bersyarat kemunculan teks sandi adalah:
$$P(C = y \mid P = x) = P(K = (y - x) \bmod 26) = \frac{1}{26}$$
Peluang marjinal teks sandi:
$$P(C = y) = \sum_{x \in \mathcal{P}} P(P = x) P(C = y \mid P = x) = \frac{1}{26} \sum_{x \in \mathcal{P}} P(P = x) = \frac{1}{26}$$
Berdasarkan Teorema Bayes:
$$P(P = x \mid C = y) = \frac{P(C = y \mid P = x) P(P = x)}{P(C = y)} = \frac{\frac{1}{26} P(P = x)}{\frac{1}{26}} = \mathbf{P(P = x)}$$

Mengamati huruf ciphertext $y$ sama sekali **tidak mengubah probabilitas** huruf aslinya. Peluang *a posteriori* sama persis dengan peluang *a priori*. Sistem terbukti **Aman Sempurna**!


### ❓ Pertanyaan 3: Untuk Menunjukkan Suatu Sandi Tidak Aman Sempurna, Apakah Cukup dengan Menunjukkan $|\mathcal{K}| < |\mathcal{P}|$?
---

### Jawaban: **YA, SANGAT CUKUP!** (Dengan Asumsi Standar $P(x) > 0, \forall x \in \mathcal{P}$)

#### 1. Landasan Teorema Shannon (Kontraposisi)
Teorema Batas Fundamental Shannon (1949) menyatakan:
$$\text{Jika suatu kriptosistem aman sempurna dan } P(P = x) > 0 \text{ untuk semua } x \in \mathcal{P}, \text{ maka wajib } |\mathcal{K}| \ge |\mathcal{P}|.$$

Dalam logika matematika, implikasi $A \implies B$ secara ekuivalen bermakna $\neg B \implies \neg A$ (**Kontraposisi**):
$$\mathbf{|\mathcal{K}| < |\mathcal{P}| \implies \text{Kriptosistem Tersebut PASTI TIDAK AMAN SEMPURNA!}}$$

Jadi, dalam menjawab soal UTS, Anda **cukup membuktikan bahwa $|\mathcal{K}| < |\mathcal{P}|$**. Ini adalah argumen formal yang sah dan lengkap tanpa perlu menurunkan rumus probabilitas yang panjang!

---

#### 2. Mengapa $|\mathcal{K}| < |\mathcal{P}|$ Langsung Menggagalkan Keamanan Sempurna? (Prinsip Sarang Merpati / Pigeonhole Principle)

Mari kita pahami intuisi logisnya:
1. Misalkan musuh mencegat ciphertext tertentu $y \in \mathcal{C}$.
2. Musuh mencoba mendekripsi $y$ dengan **seluruh kemungkinan kunci** yang ada di $\mathcal{K}$.
   Himpunan seluruh kemungkinan plaintext hasil dekripsi dari $y$ adalah:
   $$\mathcal{P}(y) = \{d_k(y) : k \in \mathcal{K}\}$$
3. Karena setiap kunci $k$ menghasilkan tepat 1 plaintext, banyaknya plaintext yang bisa didekripsi dari $y$ **paling banyak sebanyak kunci yang ada**:
   $$|\mathcal{P}(y)| \le |\mathcal{K}|$$
4. Jika $|\mathcal{K}| < |\mathcal{P}|$, maka:
   $$|\mathcal{P}(y)| < |\mathcal{P}|$$
   Artinya, himpunan $\mathcal{P}(y)$ adalah *subset sejati* dari $\mathcal{P}$.
5. **Konsekuensi Fatal:** Pasti terdapat setidaknya satu plaintext $x_0 \in \mathcal{P}$ yang **TIDAK MUNGKIN** didekripsi dari $y$ ($x_0 \notin \mathcal{P}(y)$).
6. Untuk pesan $x_0$ tersebut:
   $$P(P = x_0 \mid C = y) = \mathbf{0}$$
   Padahal probabilitas awalnya sebelum sandi dicegat adalah $P(P = x_0) > 0$!
7. Karena:
   $$P(P = x_0 \mid C = y) = 0 \neq P(P = x_0) > 0$$
   Maka definisi keamanan sempurna $P(x \mid y) = P(x)$ langsung **gugur seketika**!

#### Contoh Kasus UTS:
- **Sandi Geser untuk 2 Huruf:** $|\mathcal{K}| = 26$, sedangkan $|\mathcal{P}| = 26^2 = 676$.
  Karena $|\mathcal{K}| = 26 < 676 = |\mathcal{P}|$, maka Sandi Geser 2 huruf **pasti tidak aman sempurna**. Selesai!


### ❓ Pertanyaan 4: Kenapa Sandi Substitusi 1 Karakter Bisa Aman Sempurna? (Penjelasan Singkat & Intuitif)
---

### Jawaban Singkat, Padat, & Intuitif:

Kunci dari keamanan sempurna adalah: **apakah ciphertext yang dicegat membocorkan petunjuk tentang identitas plaintext?**

Bayangkan Anda adalah penyerang yang mencegat **1 huruf sandi saja**, misalnya huruf `'Q'`.
Pertanyaannya: dari huruf plaintext apa huruf `'Q'` ini berasal?
- Apakah dari `'A'`? Bisa, jika kunci permutasinya memetakan $\pi(\text{'A'}) = \text{'Q'}$.
- Apakah dari `'B'`? Bisa, jika $\pi(\text{'B'}) = \text{'Q'}$.
- Apakah dari `'Z'`? Bisa, jika $\pi(\text{'Z'}) = \text{'Q'}$.

---

#### Bukti Matematis Sederhana (Hanya 3 Langkah):

1. **Total Kunci Permutasi:**
   Ada $|\mathcal{K}| = 26!$ kemungkinan cara mengacak alfabet.
2. **Kunci yang Memetakan Huruf Tertentu $x$ ke Huruf Sandi $y$:**
   Jika kita tetapkan bahwa $\pi(x) = y$, maka 25 huruf sisanya bebas dipasangkan ke 25 huruf sandi sisanya.
   Banyaknya permutasi yang memenuhi $\pi(x) = y$ adalah tepat:
   $$25!$$
3. **Peluang Enkripsi:**
   Karena setiap kunci dipilih seragam dengan peluang $\frac{1}{26!}$, maka:
   $$P(C = y \mid P = x) = \frac{\text{jumlah kunci yang memetakan } x \to y}{\text{total seluruh kunci}} = \frac{25!}{26!} = \frac{25!}{26 \times 25!} = \mathbf{\frac{1}{26}}$$

---

#### Kesimpulan Inti:
Perhatikan bahwa nilai $\mathbf{\frac{1}{26}}$ ini **sama persis untuk SETIAP huruf $x$ apa pun**, dan **sama persis untuk SETIAP huruf $y$ apa pun**!

Artinya, bagi penyerang yang melihat huruf sandi `'Q'`, huruf aslinya memiliki peluang yang persis sama untuk berupa `'A'`, `'B'`, `'C'`, ..., `'Z'`. Mengetahui `'Q'` sama sekali **tidak memberi keuntungan informasi apa pun** mengenai plaintext aslinya.

> **Catatan Krusial:**
> *Mengapa sandi substitusi menjadi tidak aman pada teks panjang ($n \ge 2$)?*
> Karena begitu ada 2 huruf atau lebih, muncul **pengulangan huruf** ($x_1 = x_2 \implies y_1 = y_2$) dan pola frekuensi kata (misal huruf vokal sering muncul). Namun, untuk **1 huruf tunggal**, fenomena pengulangan ini mustahil terjadi, sehingga sandi substitusi 1 karakter **aman sempurna**!


In [ ]:
# =========================================================================
# Verifikasi Komputasi: Teorema Shannon, Batas |K| < |P|, & Substitusi 1 Karakter
# =========================================================================

# 1. Verifikasi Sandi Geser 1 Karakter (P(y|x) = 1/26 untuk semua x, y)
def shift_verify_1_char():
    # Misal distribusi teks asal bahasa Inggris tidak seragam (huruf E = 0.12, Z = 0.001, dst.)
    import random
    # Hitung untuk semua pasangan (x, y), ada berapa kunci k yang menghubungkan
    matrix = [[0]*26 for _ in range(26)]
    for x in range(26):
        for y in range(26):
            # cari k sedemikian hingga (x + k) % 26 == y
            k = (y - x) % 26
            matrix[x][y] = 1 # tepat 1 kunci!
    print("Sandi Geser 1 Karakter: Semua pasangan (x, y) memiliki tepat", matrix[0][0], "kunci unik.")
    print("P(C = y | P = x) = 1/26 = 0.03846 untuk SEMUA pasangan! -> AMAN SEMPURNA.\n")

shift_verify_1_char()

# 2. Verifikasi Batas Shannon |K| < |P| pada Sandi Geser 2 Karakter
# |K| = 26, |P| = 26^2 = 676
# Tunjukkan bahwa ada plaintext yang TIDAK MUNGKIN dihasilkan dari ciphertext y
def shift_verify_2_chars():
    # Misalkan ciphertext yang dicegat adalah y = 'AB' = (0, 1)
    y = (0, 1)
    # Himpunan plaintext yang MUNGKIN didekripsi dari y dengan 26 kemungkinan kunci k:
    possible_plaintexts = set()
    for k in range(26):
        p1 = (y[0] - k) % 26
        p2 = (y[1] - k) % 26
        possible_plaintexts.add((p1, p2))
    
    print("=== Sandi Geser 2 Karakter (|K|=26, |P|=676) ===")
    print(f"Banyaknya plaintext yang dapat didekripsi dari ciphertext 'AB': {len(possible_plaintexts)}")
    print(f"Banyaknya plaintext yang MUSTAHIL didekripsi dari 'AB'    : {676 - len(possible_plaintexts)}")
    
    sample_impossible = (0, 0) # 'AA'
    print(f"Apakah plaintext 'AA' mungkin menghasilkan 'AB'? {sample_impossible in possible_plaintexts}")
    print("P(P = 'AA' | C = 'AB') = 0, padahal P(P = 'AA') > 0!")
    print("Terbukti: |K| < |P| secara otomatis menyebabkan P(x|y) = 0 != P(x) -> GAGAL AMAN SEMPURNA!\n")

shift_verify_2_chars()

# 3. Verifikasi Sandi Substitusi 1 Karakter
import math

total_kunci_substitusi = math.factorial(26)
kunci_memetakan_x_ke_y = math.factorial(25)
peluang = kunci_memetakan_x_ke_y / total_kunci_substitusi

print("=== Sandi Substitusi 1 Karakter ===")
print(f"Total seluruh kunci permutasi |K| = 26! = {total_kunci_substitusi}")
print(f"Banyak kunci yang memetakan huruf x ke y = 25! = {kunci_memetakan_x_ke_y}")
print(f"P(C = y | P = x) = 25! / 26! = 1 / 26 = {peluang:.6f}")
print("P(C = y | P = x) bernilai konstan 1/26 untuk SETIAP x dan y -> AMAN SEMPURNA!")


### ❓ Pertanyaan Berikutnya: (Tuliskan pertanyaan di sini)
*Catatan pembahasan teori...*


In [ ]:
# Tuliskan kode eksperimen / verifikasi perhitungan di sini
pass
